# Jersey number recognizer — GSR production (maxconf)

Frozen configuration, no diagnostics: legibility > 0.9, det > 0.52, rule
maxconf (all three are the harness defaults in this package). One GPU pass,
one CPU merge, four metrics: trk_acc, numbered, -1 F1, roi kept.

Reference from the six-rule comparison run (2026-08-02, same split, same
gates): maxconf trk_acc 84.20%, numbered 85.01%, -1 F1 0.83, roi kept 57.2%.
The metrics cell compares against these — informational, exact match expected
because the worker pass is seeded and the merge is deterministic.

## Environment

Kaggle's notebook editor cannot switch the kernel to a custom venv, so every
pipeline command runs through the venv's own interpreter (PYBIN). The
bootstrap also verifies the package against MANIFEST.sha256 before anything
runs.

In [ ]:
import glob, os, subprocess, sys, zipfile

VENV = "/kaggle/tmp/venv_jn"
WORK = "/kaggle/working"
ROOT = "/kaggle/input"


def find_code(root=ROOT, work=WORK):
    """Locate run_eval.py under /kaggle/input at ANY depth.

    Kaggle mounts an uploaded archive in more than one shape depending on how it
    was added: the dataset root may BE the code folder, or it may contain the
    zip's own top-level directory, or a directory named after the archive with
    that directory inside it, or occasionally the un-extracted .zip. A
    fixed-depth glob only covers one of those, which is exactly how this cell
    failed before. Search recursively, expand a zip if that is all there is, and
    if nothing matches, SHOW what is mounted instead of asserting into a void.
    """
    def scan(base):
        return sorted((p for p in glob.glob(f"{base}/**/run_eval.py",
                                            recursive=True)
                       if "__pycache__" not in p),
                      key=lambda p: (p.count(os.sep), p))

    found = scan(root)
    if not found:
        for z in sorted(glob.glob(f"{root}/**/*.zip", recursive=True)):
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("run_eval.py") for n in zf.namelist()):
                    continue
                dest = os.path.join(work, "_unzipped")
                print(f"[bootstrap] dataset is an un-extracted archive; "
                      f"expanding {z} -> {dest}")
                zf.extractall(dest)
            found = scan(dest)
            if found:
                break

    if not found:
        print("[bootstrap] run_eval.py not found anywhere. What IS mounted:")
        if not os.path.isdir(root):
            print(f"    {root} does not exist -- no dataset attached.")
        elif not os.listdir(root):
            print(f"    {root} is empty -- no dataset attached.")
        else:
            shown = 0
            for dirpath, dirs, files in os.walk(root):
                dirs[:] = [d for d in dirs if d != "__pycache__"]
                depth = dirpath[len(root):].count(os.sep)
                if depth > 3:
                    dirs[:] = []
                    continue
                print(f"    {'  ' * depth}{os.path.basename(dirpath) or root}/")
                for f in sorted(files)[:8]:
                    print(f"    {'  ' * (depth + 1)}{f}")
                if len(files) > 8:
                    print(f"    {'  ' * (depth + 1)}... +{len(files) - 8} more")
                shown += 1
                if shown > 40:
                    print("    ...")
                    break
        raise SystemExit(
            "[bootstrap] Attach the pipeline zip via '+ Add Input' -> Datasets, "
            "then re-run this cell. If it IS attached, the listing above shows "
            "where it landed and this cell will find it at any depth.")

    if len({os.path.dirname(p) for p in found}) > 1:
        print("[bootstrap] WARNING: multiple copies attached -- using the "
              "shallowest. Detach the stale ones:")
        for p in found:
            print("   ", p)
    return os.path.dirname(found[0])


SRC = find_code()
print("code:", SRC)

CODE = os.path.join(WORK, "jn_pipeline")
subprocess.run(["rm", "-rf", CODE], check=False)
subprocess.run(["cp", "-r", SRC, CODE], check=True)
os.chdir(CODE)
print("cwd :", os.getcwd())
assert os.path.exists("run_eval.py") and os.path.exists("MANIFEST.sha256")


# Kaggle's kernel presets MPLBACKEND to an inline backend whose module is NOT in
# the pipeline venv; matplotlib raises ValueError at IMPORT time, and
# torchmetrics (via pytorch-lightning) imports matplotlib. Every `!$PY ...`
# child inherits this environ, so setting it once here covers all of them.
# The scripts each guard themselves too -- this is the belt to that pair of
# braces, and it is what keeps a failure from looking like a build failure.
os.environ["MPLBACKEND"] = "Agg"


def sh(cmd):
    print("$", cmd, flush=True)
    rc = subprocess.run(cmd, shell=True,
                        env=dict(os.environ, MPLBACKEND="Agg")).returncode
    if rc != 0:
        raise SystemExit(f"FAILED (rc={rc}): {cmd}")


# Every shipped file, hash-checked, before anything else runs.
sh("sha256sum -c MANIFEST.sha256 | grep -v ': OK$' ; "
   "echo \"[bootstrap] $(sha256sum -c MANIFEST.sha256 2>/dev/null "
   "| grep -c ': OK$') files verified\"")

sh(f"{sys.executable} setup_kaggle.py --venv {VENV}")
PYBIN = f"{VENV}/bin/python"
print("PYBIN:", PYBIN)

## Weights (hash-checked)

In [ ]:
sh(f"{PYBIN} fetch_weights.py --out-dir models")
sh(f"{PYBIN} stage_weights.py --out models/koshkina_sn_parseq.ckpt")

## GSR-2024 test — stage

In [ ]:
SPLIT = "test"
DATA_GSR = "/kaggle/tmp/data/gsr"
sh(f"{PYBIN} stage_data.py --root {DATA_GSR} --splits {SPLIT} --delete-zip")

## GPU pass (both GPUs, sharded; resumable via the cache)

In [ ]:
sh(f"python dual_gpu.py --gpus 0,1 -- {PYBIN} run_eval.py "
   f"--root {DATA_GSR} --split {SPLIT} --cache work/eval_cache/{SPLIT} "
   f"--ckpt models/best_icdar_hmean_epoch_10.pth "
   f"--parseq models/koshkina_sn_parseq.ckpt "
   f"--legibility-weights models/sn_legibility.pth")

## Merge — frozen config

No flags beyond the defaults (leg 0.9, det 0.52, maxconf); passed explicitly
anyway so the log is self-describing.

In [ ]:
sh(f"{PYBIN} run_eval.py --root {DATA_GSR} --split {SPLIT} "
   f"--cache work/eval_cache/{SPLIT} --merge --legibility-thr 0.9 "
   f"--det-thr 0.52 --rule maxconf --out work/results_{SPLIT}.json")

## Metrics

In [ ]:
import json

REF = {"trk_acc": 84.20, "numbered": 85.01, "minus1_f1": 0.83, "roi_kept": 57.2}

r = json.load(open(f"work/results_{SPLIT}.json"))
got = {"trk_acc": round(100 * r["trk_acc"], 2),
       "numbered": round(100 * r["numbered"], 2),
       "minus1_f1": round(r["minus1_f1"], 2),
       "roi_kept": round(100 * r["roi_kept"], 1)}
print(f"[GSR-2024 {SPLIT}] frozen config: leg > 0.9, det > 0.52, maxconf")
print(f"            trk_acc  numbered   -1 F1  roi kept")
print(f"got          {got['trk_acc']:6.2f}%   {got['numbered']:6.2f}%"
      f"    {got['minus1_f1']:4.2f}     {got['roi_kept']:4.1f}%")
print(f"reference    {REF['trk_acc']:6.2f}%   {REF['numbered']:6.2f}%"
      f"    {REF['minus1_f1']:4.2f}     {REF['roi_kept']:4.1f}%")
ok = all(abs(got[k] - REF[k]) < 1e-9 for k in REF)
print("MATCHES REFERENCE" if ok else
      "DIFFERS FROM REFERENCE -- deterministic path, investigate before "
      "integrating")